In [165]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [166]:
df=pd.read_csv('examples.csv')
df['sqft']=df['sqft'].str.replace('sqft','')
df['land']=df['land'].str.replace('perches','')
df

,price,lat,long,Bathroom,Bedroom,sqft,land
0,65000000,6.916891,79.968552,4,5,3600.0,10.0
1,23000000,6.799951,79.923117,1,2,1400.0,8.0
2,35000000,6.932625,79.890314,4,4,2200.0,6.0
3,45000000,6.851279,79.865977,5,5,2500.0,4.96
4,26000000,6.875005,79.900833,3,4,2600.0,9.5
...,...,...,...,...,...,...,...
1225,13500000,6.937244,79.896214,2,2,750.0,3.0
1226,8500000,6.937244,79.896214,2,3,800.0,3.5
1227,16800000,6.939743,79.902341,2,4,900.0,3.0
1228,15000000,6.937244,79.896214,2,4,900.0,3.0


In [167]:
df=df.astype(float)
df

,price,lat,long,Bathroom,Bedroom,sqft,land
0,65000000.0,6.916891,79.968552,4.0,5.0,3600.0,10.00
1,23000000.0,6.799951,79.923117,1.0,2.0,1400.0,8.00
2,35000000.0,6.932625,79.890314,4.0,4.0,2200.0,6.00
3,45000000.0,6.851279,79.865977,5.0,5.0,2500.0,4.96
4,26000000.0,6.875005,79.900833,3.0,4.0,2600.0,9.50
...,...,...,...,...,...,...,...
1225,13500000.0,6.937244,79.896214,2.0,2.0,750.0,3.00
1226,8500000.0,6.937244,79.896214,2.0,3.0,800.0,3.50
1227,16800000.0,6.939743,79.902341,2.0,4.0,900.0,3.00
1228,15000000.0,6.937244,79.896214,2.0,4.0,900.0,3.00


In [168]:
from sklearn.preprocessing import StandardScaler
import joblib
def scale_dataset(df,fit_scaler):
    x=df[df.columns[3:]].values
    y=df['price'].values
    lat_long=df[['lat','long']].values
    if fit_scaler:
        scaler=StandardScaler()
        x=scaler.fit_transform(x)
        joblib.dump(scaler,'standard_scaler.joblib')
    else:
        scaler=joblib.load('standard_scaler.joblib')
        x=scaler.transform(x)
    return x,y,lat_long

In [169]:
train,test=np.split(df.sample(frac=1),[int(len(df)*0.8)])

/media/kavindu/54AAAD8CAAAD6AE6/Linux/pyenv/venv/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [170]:
x_train,y_train,lat_long_train=scale_dataset(train,True)
x_test,y_test,lat_long_test=scale_dataset(test,False)

In [ ]:
from sklearn.metrics import pairwise_distances
from sklearn.linear_model import LinearRegression
class mymodel1:
    '''
    this model get x(feature vector with n samples) y(n samples,) and 
    neighbour_array to idnetify neighbours.
    it will create weight_array from neighbour_array by using inverse to distance.
    '''
    def __init__(self):
        pass
    def fit(self,x,y,neighbour):
        self.x=x
        self.y=y.reshape(-1,1)
        self.neighbour=neighbour
    def predict(self,x_predict,neighbour_predict):
        distances=pairwise_distances(neighbour_predict,self.neighbour)
        y_predict=np.zeros((len(x_predict),1))
        for i in range(len(x_predict)):
            distance_array=distances[i].reshape(-1,1)
            distance_array=distance_array/distance_array.max(axis=0)
            if distance_array.min(axis=0)==0:
                zero_indexes=np.where(distance_array==0)[0]
                distance_array[zero_indexes]=2
                second_min=distance_array.min(axis=0)
                distance_array[zero_indexes]=second_min/1000
            weight_array=1/distance_array
            regression=LinearRegression()
            regression.fit(self.x,self.y,weight_array.reshape(-1,))
            y_predict[i]=regression.predict(x_predict[i].reshape(1,-1))
        return y_predict

In [172]:
def accuracy(Y_real,Y_predict,range_):
    Y_real=Y_real.reshape(-1,)
    Y_predict=Y_predict.reshape(-1,)
    acc=0
    for i in range(len(Y_real)):
        min_range=Y_real[i]*(1-range_)
        max_range=Y_real[i]*(1+range_)
        if min_range<Y_predict[i]<max_range:
            acc+=1
    return acc/len(Y_predict)*100

In [173]:
model=mymodel1()
model.fit(x_train,y_train,lat_long_train)
y_predict=model.predict(x_test,lat_long_test)
accuracy(y_test,y_predict,0.2)

44.3089430894309

In [174]:
from sklearn.neighbors import NearestNeighbors
class mymodel2:
    def __init__(self):
        pass
    def fit(self,x,y,neighbour):
        self.x=x
        self.y=y.reshape(-1,1)
        self.classifier=NearestNeighbors(radius=5/111)
        self.classifier.fit(neighbour)
    def predict(self,x_predict,neighbour_predict):
        y_predict=np.zeros((len(x_predict),1))
        all_circle_indices=self.classifier.radius_neighbors(neighbour_predict,5/111,False)
        for i in range(len(x_predict)):
            circle_indices=all_circle_indices[i]
            if len(circle_indices)==0:
                print('can\'t predict, no training data inside 5km radius')
                continue
            x=self.x[circle_indices]
            y=self.y[circle_indices]
            regression=LinearRegression()
            regression.fit(x,y)
            y_predict[i]=regression.predict(x_predict[i].reshape(1,-1))
        return y_predict

In [175]:
model=mymodel2()
model.fit(x_train,y_train,lat_long_train)
y_predict=model.predict(x_test,lat_long_test)
accuracy(y_test,y_predict,0.2)

can't predict, no training data inside 5km radius
can't predict, no training data inside 5km radius
can't predict, no training data inside 5km radius
can't predict, no training data inside 5km radius
can't predict, no training data inside 5km radius


31.300813008130078

In [190]:
class mymodel3:
    def __init__(self):
        pass
    def fit(self,x,y,neighbour):
        self.x=x
        self.y=y.reshape(-1,1)
        self.neighbour=neighbour
        self.classifier=NearestNeighbors(n_neighbors=100)
        self.classifier.fit(neighbour)
    def predict(self,x_predict,neighbour_predict):
        distances=pairwise_distances(neighbour_predict,self.neighbour)
        y_predict=np.zeros((len(x_predict),1))
        nearest_k=self.classifier.kneighbors(neighbour_predict,300,False)
        for i in range(len(x_predict)):
            k_indices=nearest_k[i]
            x=self.x[k_indices]
            y=self.y[k_indices]
            distance_array=distances[i].reshape(-1,1)[k_indices]
            distance_array=distance_array/distance_array.max(axis=0)
            if distance_array.min(axis=0)==0:
                zero_indexes=np.where(distance_array==0)[0]
                distance_array[zero_indexes]=2
                second_min=distance_array.min(axis=0)
                distance_array[zero_indexes]=second_min/1000
            weight_array=1/distance_array
            regression=LinearRegression()
            regression.fit(x,y,weight_array.reshape(-1,))
            y_predict[i]=regression.predict(x_predict[i].reshape(1,-1))
        return y_predict

In [191]:
model=mymodel3()
model.fit(x_train,y_train,lat_long_train)
y_predict=model.predict(x_test,lat_long_test)
accuracy(y_test,y_predict,0.2)

43.49593495934959